# 02 · Text Preprocessing Pipeline

**Goal:** Clean and normalise raw texts, apply symmetric masking,
tokenize via spaCy, and save the preprocessed dataset.

**Sections:**
1. Load raw data
2. Define cleaning functions
3. Apply preprocessing pipeline
4. Before / after comparison
5. spaCy tokenization & linguistic annotations
6. Save preprocessed dataset
7. Validation checks

In [ ]:
# ── 0. Imports & settings ────────────────────────────────────────────────

import json
import pathlib
import re
import time
import warnings
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.bbox"] = "tight"

# ── Paths ──
def find_project_root(start: pathlib.Path | None = None) -> pathlib.Path:
    """Find project root robustly for Jupyter / VS Code / scripts."""
    start = (start or pathlib.Path.cwd()).resolve()
    candidates = [start, *start.parents]
    markers = [
        ("data", "final"),
        ("notebooks",),
        ("outputs",),
    ]
    for path in candidates:
        if all((path / part).exists() for part in ("data",)) and (path / "data" / "final").exists():
            return path
    # fallback compatible with notebook inside notebooks/
    if (start.parent / "data" / "final").exists():
        return start.parent
    return start

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "final"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "figures" / "preprocessing"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT  : {PROJECT_ROOT}")
print(f"DATA_DIR      : {DATA_DIR}")
print(f"PROCESSED_DIR : {PROCESSED_DIR}")
print(f"Files found   : {sorted(p.name for p in DATA_DIR.iterdir()) if DATA_DIR.exists() else 'DIR NOT FOUND'}")

## 1 · Load raw data

In [ ]:
def load_dataset(data_dir: pathlib.Path) -> pd.DataFrame:
    """Auto-detect format and load all splits into a single DataFrame."""
    data_dir = pathlib.Path(data_dir)

    split_files = {s: data_dir / f"{s}.parquet" for s in ["train", "val", "test"]}
    if all(f.exists() for f in split_files.values()):
        frames = []
        for split_name, fpath in split_files.items():
            tmp = pd.read_parquet(fpath)
            if "split" not in tmp.columns:
                tmp["split"] = split_name
            frames.append(tmp)
        df = pd.concat(frames, ignore_index=True)
        print(f"Loaded 3 parquet splits → {len(df):,} rows")
        return df

    single = data_dir / "dataset.parquet"
    if single.exists():
        df = pd.read_parquet(single)
        print(f"Loaded single parquet → {len(df):,} rows")
        return df

    csv_files = {s: data_dir / f"{s}.csv" for s in ["train", "val", "test"]}
    if all(f.exists() for f in csv_files.values()):
        frames = []
        for split_name, fpath in csv_files.items():
            tmp = pd.read_csv(fpath)
            if "split" not in tmp.columns:
                tmp["split"] = split_name
            frames.append(tmp)
        df = pd.concat(frames, ignore_index=True)
        print(f"Loaded 3 CSV splits → {len(df):,} rows")
        return df

    raise FileNotFoundError(
        f"No dataset found in {data_dir}. "
        "Expected train/val/test.parquet, dataset.parquet, or train/val/test.csv."
    )


df = load_dataset(DATA_DIR)

REQUIRED = ["text", "label", "split"]
for col in REQUIRED:
    assert col in df.columns, f"Missing required column: {col}"

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
display(df.head(2))

In [ ]:
# Keep original text for comparison
df["text_raw"] = df["text"].fillna("").astype(str)
df["text"] = df["text"].fillna("").astype(str)

print(f"Null texts: {df['text'].isnull().sum()}")
print(f"Empty texts: {(df['text'].str.strip() == '').sum()}")

## 2 · Define cleaning functions

**Key design principles:**
- URL masking is **symmetric** across all groups (A, B, C, D)
- Email masking is symmetric
- We preserve case (transformers are case-sensitive; classical models may lowercase later)
- Minimal destructive cleaning — we want to preserve stylometric signal

In [ ]:
# ── 2a. Regex patterns ──

# URL pattern (covers http, https, www, and common TLDs)
RE_URL = re.compile(
    r"https?://[^\s<>\"']+|www\.[^\s<>\"']+"
    r"|(?<!@)\b[a-zA-Z0-9._%+-]+\.(?:com|org|net|edu|gov|io|co|uk|ru|de|fr|info|biz)\b(?:/[^\s<>\"']*)?",
    re.IGNORECASE,
)

# Email pattern
RE_EMAIL = re.compile(
    r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b",
    re.IGNORECASE,
)

# Phone pattern (simple: sequences of digits with optional separators)
RE_PHONE = re.compile(
    r"(?:\+?\d{1,3}[-.\s]?)?(?:\(?\d{2,4}\)?[-.\s]?){2,4}\d{2,4}"
)

# Repeated whitespace
RE_MULTI_SPACE = re.compile(r"[ \t]+")
RE_MULTI_NEWLINE = re.compile(r"\n{3,}")

# HTML tags (minimal — some phishing emails have residual HTML)
RE_HTML = re.compile(r"<[^>]{1,200}>")

# Non-printable / control characters (keep newlines)
RE_CONTROL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]")

print("Regex patterns compiled.")

In [ ]:
# ── 2b. Cleaning pipeline ──

def clean_text(
    text: str,
    mask_url: bool = True,
    mask_email: bool = True,
    mask_phone: bool = False,
    strip_html: bool = True,
    normalize_whitespace: bool = True,
    remove_control_chars: bool = True,
) -> str:
    """Apply symmetric cleaning pipeline to a single text.

    IMPORTANT: All masking operations are applied identically
    to ALL groups (A, B, C, D) to avoid information leakage.
    """
    if not isinstance(text, str):
        return ""

    # 1. Remove control characters
    if remove_control_chars:
        text = RE_CONTROL.sub("", text)

    # 2. Strip HTML tags
    if strip_html:
        text = RE_HTML.sub(" ", text)

    # 3. Mask emails first (to avoid partial URL replacement inside emails)
    if mask_email:
        text = RE_EMAIL.sub("[EMAIL]", text)

    # 4. Mask URLs (symmetric across all groups)
    if mask_url:
        text = RE_URL.sub("[URL]", text)

    # 5. Mask phone numbers (optional)
    if mask_phone:
        text = RE_PHONE.sub("[PHONE]", text)

    # 6. Normalize whitespace
    if normalize_whitespace:
        text = RE_MULTI_NEWLINE.sub("\n\n", text)
        text = RE_MULTI_SPACE.sub(" ", text)

    # 7. Strip leading/trailing whitespace
    text = text.strip()

    return text


# Quick test
test_cases = [
    "Visit https://phishing-site.com/login or email admin@evil.org now!!!",
    "<html><body>Dear customer,  \x00\x01  click here</body></html>",
    "Call us at +1 (555) 123-4567 or visit www.bank.com/verify",
    "Normal text with    extra   spaces\n\n\n\n\nand newlines.",
]

for t in test_cases:
    print(f"IN:  {t!r}")
    print(f"OUT: {clean_text(t)!r}")
    print()

## 3 · Apply preprocessing pipeline

In [ ]:
# ── 3a. Apply cleaning ──
t0 = time.time()
df["text"] = df["text_raw"].apply(clean_text)
elapsed = time.time() - t0
speed = len(df) / elapsed if elapsed > 0 else np.nan
print(f"Cleaning applied to {len(df):,} texts in {elapsed:.1f}s ({speed:.0f} texts/s)")

In [ ]:
# ── 3b. Detect empty texts after cleaning ──
empty_mask = df["text"].str.strip() == ""
n_empty = int(empty_mask.sum())
print(f"Empty texts after cleaning: {n_empty}")

if n_empty > 0:
    print("\nSample of originally non-empty texts that became empty:")
    became_empty = empty_mask & (df["text_raw"].str.strip() != "")
    if became_empty.any():
        display(df.loc[became_empty, ["text_raw", "label", "split"]].head(5))

In [ ]:
# ── 3c. Masking statistics ──
n_url = int(df["text"].str.count(re.escape("[URL]")).sum())
n_email = int(df["text"].str.count(re.escape("[EMAIL]")).sum())

print(f"Total [URL]   tokens inserted: {n_url:,}")
print(f"Total [EMAIL] tokens inserted: {n_email:,}")

# Per-label breakdown to verify symmetry of the PIPELINE application.
# Counts do not have to match across classes because source corpora differ.
LABEL_MAP = {0: "Human", 1: "LLM"}
df["label_str"] = df["label"].map(LABEL_MAP).fillna(df["label"].astype(str))

mask_stats = df.groupby("label_str").apply(
    lambda g: pd.Series({
        "texts_with_URL": int((g["text"].str.contains(re.escape("[URL]"), regex=True)).sum()),
        "texts_with_EMAIL": int((g["text"].str.contains(re.escape("[EMAIL]"), regex=True)).sum()),
        "total": int(len(g)),
    })
)
mask_stats["pct_URL"] = (mask_stats["texts_with_URL"] / mask_stats["total"] * 100).round(1)
mask_stats["pct_EMAIL"] = (mask_stats["texts_with_EMAIL"] / mask_stats["total"] * 100).round(1)
print("\nMasking distribution:")
display(mask_stats)

## 4 · Before / after comparison

In [ ]:
# ── 4a. Length change analysis ──
df["n_chars_raw"] = df["text_raw"].str.len()
df["n_chars_clean"] = df["text"].str.len()
df["chars_diff"] = df["n_chars_clean"] - df["n_chars_raw"]
df["chars_diff_pct"] = (df["chars_diff"] / df["n_chars_raw"].replace(0, np.nan) * 100).round(2)

print("Character-count change after preprocessing:")
print(df["chars_diff"].describe().round(1))
print(f"\nMean % change: {df['chars_diff_pct'].mean():.2f}%")

In [ ]:
# ── 4b. Histogram of character-count change ──
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Absolute change
ax = axes[0]
ax.hist(df["chars_diff"].clip(-500, 100), bins=80, color="steelblue", edgecolor="white", linewidth=0.3)
ax.axvline(0, color="red", linestyle="--", linewidth=0.8)
ax.set_xlabel("Character count change (clean − raw)")
ax.set_ylabel("Frequency")
ax.set_title("Absolute change")

# Percentage change
ax = axes[1]
valid_pct = df["chars_diff_pct"].dropna().clip(-50, 10)
ax.hist(valid_pct, bins=80, color="coral", edgecolor="white", linewidth=0.3)
ax.axvline(0, color="red", linestyle="--", linewidth=0.8)
ax.set_xlabel("% change in character count")
ax.set_ylabel("Frequency")
ax.set_title("Relative change (%)")

fig.suptitle("Text length change after preprocessing", y=1.02, fontweight="bold")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "length_change_hist.png")
plt.show()

In [ ]:
# ── 4c. Side-by-side examples ──
# Show texts with largest absolute change
top_changed = df.nlargest(5, "chars_diff", keep="first")
bot_changed = df.nsmallest(5, "chars_diff", keep="first")

print("=" * 70)
print("TOP 5: Texts that GREW after cleaning (unusual — investigate)")
print("=" * 70)
for _, row in top_changed.iterrows():
    print(f"\n[label={row['label']}, split={row['split']}, Δ={int(row['chars_diff']):+d}]")
    print(f"  RAW:   {row['text_raw'][:200]}...")
    print(f"  CLEAN: {row['text'][:200]}...")

print("\n" + "=" * 70)
print("TOP 5: Texts that SHRANK the most")
print("=" * 70)
for _, row in bot_changed.iterrows():
    print(f"\n[label={row['label']}, split={row['split']}, Δ={int(row['chars_diff']):+d}]")
    print(f"  RAW:   {row['text_raw'][:200]}...")
    print(f"  CLEAN: {row['text'][:200]}...")

## 5 · spaCy tokenization & linguistic annotations

In [ ]:
# ── 5a. Load spaCy ──
import spacy

# Use sm model for speed; features notebook will use the same
MODEL_NAME = "en_core_web_sm"

try:
    nlp = spacy.load(MODEL_NAME, disable=["ner"])  # NER not needed for preprocessing
except OSError:
    print(f"Downloading {MODEL_NAME}...")
    spacy.cli.download(MODEL_NAME)
    nlp = spacy.load(MODEL_NAME, disable=["ner"])

# Increase max length for long emails
nlp.max_length = 500_000
print(f"spaCy model: {MODEL_NAME}, pipeline: {nlp.pipe_names}")

In [ ]:
# ── 5b. Tokenize a sample to verify ──
sample_texts = df["text"].head(3).tolist()

for i, doc in enumerate(nlp.pipe(sample_texts, batch_size=8)):
    print(f"\n--- Sample {i} ({len(doc)} tokens) ---")
    for tok in list(doc)[:20]:
        print(f"  {tok.text:20s}  POS={tok.pos_:6s}  TAG={tok.tag_:6s}  lemma={tok.lemma_}")
    if len(doc) > 20:
        print(f"  ... ({len(doc) - 20} more tokens)")

In [ ]:
# ── 5c. Batch tokenization — extract token counts & sentence counts ──
# Full tokenization will be done in 03_features; here we just validate & get basic stats

BATCH_SIZE = 256
n_tokens_list = []
n_sents_list = []

t0 = time.time()
for doc in nlp.pipe(df["text"].tolist(), batch_size=BATCH_SIZE, n_process=1):
    n_tokens_list.append(len(doc))
    n_sents_list.append(sum(1 for _ in doc.sents))

elapsed = time.time() - t0
speed = len(df) / elapsed if elapsed > 0 else np.nan
print(f"Tokenized {len(df):,} texts in {elapsed:.1f}s ({speed:.0f} texts/s)")

df["n_tokens_spacy"] = n_tokens_list
df["n_sents_spacy"] = n_sents_list

In [ ]:
# ── 5d. Token count distribution ──
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, col, title in [
    (axes[0], "n_tokens_spacy", "spaCy token count"),
    (axes[1], "n_sents_spacy", "spaCy sentence count"),
]:
    for label_val, color, lbl in [(0, "#4C72B0", "Human"), (1, "#DD8452", "LLM")]:
        data = df.loc[df["label"] == label_val, col]
        upper = data.quantile(0.99) if len(data) else None
        ax.hist(
            data.clip(upper=upper) if upper is not None else data,
            bins=60, alpha=0.55, color=color, label=lbl,
            edgecolor="white", linewidth=0.3,
        )
    ax.set_xlabel(title)
    ax.set_ylabel("Frequency")
    ax.legend()

fig.suptitle("spaCy tokenization statistics", y=1.02, fontweight="bold")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "spacy_token_distribution.png")
plt.show()

In [ ]:
# ── 5e. Avg tokens per sentence ──
df["avg_tokens_per_sent"] = df["n_tokens_spacy"] / df["n_sents_spacy"].replace(0, np.nan)

print("Average tokens per sentence:")
print(
    df.groupby("label_str")["avg_tokens_per_sent"]
    .describe(percentiles=[0.25, 0.5, 0.75])
    .round(2)
)

## 6 · Save preprocessed dataset

In [ ]:
# ── 6a. Select columns to save ──
# Keep only the columns needed downstream; drop intermediate ones

# Columns that definitely exist
KEEP_COLS = ["text", "label", "split"]

# Optional columns — keep if present
OPTIONAL_COLS = [
    "group", "is_fraud", "content_type", "type", "category", "content_type_id",
    "source", "model", "generator", "origin", "origin_model", "dataset_source",
    "temperature_style", "temperature_value", "length_bin", "generation_type",
    "n_tokens_spacy", "n_sents_spacy", "avg_tokens_per_sent",
]
for col in OPTIONAL_COLS:
    if col in df.columns and col not in KEEP_COLS:
        KEEP_COLS.append(col)

df_save = df[KEEP_COLS].copy()
print(f"Columns to save: {KEEP_COLS}")
print(f"Shape: {df_save.shape}")

In [ ]:
# ── 6b. Save per-split parquet files ──
split_order = [s for s in ["train", "val", "test"] if s in set(df_save["split"])]

for split_name in split_order:
    split_df = df_save[df_save["split"] == split_name].reset_index(drop=True)
    out_path = PROCESSED_DIR / f"{split_name}.parquet"
    split_df.to_parquet(out_path, index=False)
    print(f"  {split_name}: {len(split_df):>6,} rows → {out_path}")

# Also save full dataset
full_path = PROCESSED_DIR / "dataset.parquet"
df_save.to_parquet(full_path, index=False)
print(f"\n  Full: {len(df_save):>6,} rows → {full_path}")

In [ ]:
# ── 6c. Save preprocessing metadata ──
meta = {
    "preprocessing_steps": [
        "remove_control_chars",
        "strip_html",
        "mask_url_symmetric ([URL])",
        "mask_email_symmetric ([EMAIL])",
        "normalize_whitespace",
        "strip",
    ],
    "spacy_model": MODEL_NAME,
    "total_samples": int(len(df_save)),
    "splits": {
        s: int((df_save["split"] == s).sum())
        for s in sorted(df_save["split"].dropna().unique())
    },
    "empty_after_cleaning": int(n_empty),
    "url_tokens_inserted": int(n_url),
    "email_tokens_inserted": int(n_email),
}

meta_path = PROCESSED_DIR / "preprocessing_meta.json"
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print(f"Metadata saved → {meta_path}")
print(json.dumps(meta, indent=2, ensure_ascii=False))

## 7 · Validation checks

In [ ]:
# ── 7a. Reload and verify ──
df_check = pd.read_parquet(PROCESSED_DIR / "dataset.parquet")
assert len(df_check) == len(df_save), "Row count mismatch!"
assert df_check["text"].isnull().sum() == 0, "Null texts found!"
print(f"✅ Reload OK: {len(df_check):,} rows, 0 nulls")

In [ ]:
# ── 7b. Label distribution preserved ──
orig_dist = df["label"].value_counts().sort_index()
check_dist = df_check["label"].value_counts().sort_index()
assert (orig_dist == check_dist).all(), "Label distribution changed!"
print("✅ Label distribution preserved")
print(check_dist)

In [ ]:
# ── 7c. Split sizes preserved ──
orig_splits = df["split"].value_counts().sort_index()
check_splits = df_check["split"].value_counts().sort_index()
assert (orig_splits == check_splits).all(), "Split sizes changed!"
print("✅ Split sizes preserved")
print(check_splits)

In [ ]:
# ── 7d. No data leakage: check no duplicates across splits ──
for s1, s2 in [("train", "val"), ("train", "test"), ("val", "test")]:
    if s1 in set(df_check["split"]) and s2 in set(df_check["split"]):
        texts_s1 = set(df_check.loc[df_check["split"] == s1, "text"])
        texts_s2 = set(df_check.loc[df_check["split"] == s2, "text"])
        overlap = texts_s1 & texts_s2
        if overlap:
            print(f"⚠️  {len(overlap)} duplicate texts between {s1} and {s2}!")
        else:
            print(f"✅ No text overlap between {s1} and {s2}")

In [ ]:
# ── 7e. Claude held-out check ──
SRC_COL = None
for candidate in ["origin_model", "dataset_source", "source", "model", "generator", "origin"]:
    if candidate in df_check.columns:
        SRC_COL = candidate
        break

if SRC_COL:
    series = df_check[SRC_COL].astype(str)
    claude_mask = series.str.lower().str.contains("claude", na=False)
    if claude_mask.any():
        claude_splits = sorted(df_check.loc[claude_mask, "split"].astype(str).unique().tolist())
        if set(claude_splits) <= {"test"}:
            print("✅ Claude is held-out (test only)")
        else:
            print(f"❌ Claude found in splits: {claude_splits} — VIOLATION!")
    else:
        print("ℹ️  No Claude entries in source column")
else:
    print("ℹ️  No source/model column — Claude check skipped")

In [ ]:
print("\n" + "=" * 60)
print("Preprocessing notebook complete.")
print(f"Processed data : {PROCESSED_DIR}")
print(f"Figures        : {OUTPUT_DIR}")
print("=" * 60)